# Library Imports and Data Saving

### Fetch the data and save it

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
from tqdm import tqdm
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from stonks_modules import *

# The ticker for State Bank of India on the National Stock Exchange
# ticker_symbols = ["^NSEI"]
ticker_symbols = ["NVDA", "AMD", "AAPL", "GOOGL"]
sbi = yf.Ticker(ticker_symbols[0])
tickers = {symbol:yf.Ticker(symbol) for symbol in ticker_symbols}


### Manipulate the data to get closes and baseline profit

In [ ]:
hourly_data_files = {ticker_symbol: fetch_and_save_data(ticker)[0] for ticker_symbol, ticker in tickers.items()}
hourly_dataframes = {ticker_symbol: pd.read_csv(filename) for ticker_symbol, filename in hourly_data_files.items()}
hourly_closes = {ticker_symbol: df[["Close"]] for ticker_symbol, df in hourly_dataframes.items()}
hourly_closes_arrays = {ticker_symbol: df["Close"].values for ticker_symbol, df in hourly_closes.items()}

In [ ]:
baseline_profits = {}

for ticker_symbol, close_df in hourly_closes.items():
    first_close = close_df.iloc[0]   # keeps same behavior as your original code (Series)
    last_close = close_df.iloc[-1]
    baseline_profits[ticker_symbol] = ((100 / first_close) * last_close).iloc[0]

# Creating the Heatmaps (for default Wait = 5)

### Initialize heatmap data

In [ ]:
sell_multipliers = np.round(np.linspace(1.01, 1.10, 90), decimals=3)
buy_multipliers = np.round(np.linspace(0.90, 0.99, 90), decimals=3)

In [ ]:
print(sell_multipliers)

In [ ]:
backtested_profits = {
    ticker_symbol : backtest_basic_dip_strategy(close_array, buy_multipliers, sell_multipliers, wait_period=11)
    for ticker_symbol, close_array in hourly_closes_arrays.items()
	}

### Make heatmaps

In [ ]:
profit_compared_to_baseline = {}
for ticker_symbol, profits_matrix in backtested_profits.items():
    profit_compared_to_baseline[ticker_symbol] = (profits_matrix - baseline_profits[ticker_symbol]) * 100 / baseline_profits[ticker_symbol]

for ticker_symbol, profits_matrix in profit_compared_to_baseline.items():
    # Creating robust string labels for the axes since Plotly interprets float arrays strictly numerically by default
    x_labels = [f"{x:.2f}" for x in sell_multipliers]
    y_labels = [f"{y:.2f}" for y in buy_multipliers]
    
    fig = go.Figure(data=go.Heatmap(
        z=profits_matrix,
        x=x_labels,
        y=y_labels,
        colorscale=[[0, "red"], [0.5, "white"], [1, "green"]],
        zmid=0, # This replaces TwoSlopeNorm by enforcing 0 is the center color (white)
        colorbar=dict(title="Profit Above Baseline (%)")
    ))
    
    fig.update_layout(
        title=f"Profit Heatmap for {ticker_symbol} (Represented as % Difference from Baseline)",
        xaxis_title="Sell Multipliers",
        yaxis_title="Buy Multipliers",
        width=800,
        height=700
    )
    
    # Decimate ticks for readability if necessary (Plotly usually handles it, but just in case)
    fig.update_xaxes(nticks=20)
    fig.update_yaxes(nticks=20)
    
    fig.show()
    
    # Optional: Save exact state to artifacts
    import os
    artifact_dir = "/Users/mridul/.gemini/antigravity/brain/ee4a65e0-e84e-48a2-ba84-be0f22e749b7"
    # fig.write_image(os.path.join(artifact_dir, f"plotly_heatmap_wait_5_{ticker_symbol}.png"))


# Creating the Heatmaps (for a list of waits)

### Initialize heatmap data

In [ ]:
wait_periods = list(range(1, 21))
sell_multipliers = np.round(np.linspace(1.01, 1.10, 90), 3)
buy_multipliers = np.round(np.linspace(0.90, 0.99, 90), 3)

In [ ]:
backtested_profits_with_wait_periods = {
    ticker_symbol : {
        wait_period: backtest_basic_dip_strategy(close_array, buy_multipliers, sell_multipliers, wait_period=wait_period)
        for wait_period in wait_periods
    }
    for ticker_symbol, close_array in hourly_closes_arrays.items()
}

### Make heatmaps

In [ ]:
from plotly.subplots import make_subplots

for ticker_symbol, backtedted_profits_by_wait in backtested_profits_with_wait_periods.items():
    baseline_profit = baseline_profits[ticker_symbol]
    all_profits = [(backtedted_profits_by_wait[w] - baseline_profit) * 100 / baseline_profit for w in wait_periods]
    vmin = min(p.min() for p in all_profits)
    vmax = max(p.max() for p in all_profits)
    
    fig = make_subplots(
        rows=5, cols=4, 
        subplot_titles=[f"Wait: {w} Hours" for w in wait_periods],
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.05,
        horizontal_spacing=0.05
    )
    
    # Generate identical labels for the ticks
    x_labels = [f"{x:.2f}" for x in sell_multipliers]
    y_labels = [f"{y:.2f}" for y in buy_multipliers]
    
    for w_idx, wait in enumerate(wait_periods):
        row = (w_idx // 4) + 1
        col = (w_idx % 4) + 1
        
        profits_matrix = (backtedted_profits_by_wait[wait] - baseline_profit) * 100 / baseline_profit
        
        # Natively, zmid=0 works on individual go.Heatmap traces, but sharing colorscales
        # across subplots requires either explicitly mapping zmin/zmax or using coloraxis.
        fig.add_trace(go.Heatmap(
            z=profits_matrix,
            x=x_labels,
            y=y_labels,
            colorscale=[[0, "red"], [0.5, "white"], [1, "green"]],
            zmid=0,
            coloraxis="coloraxis" # Connects all heatmaps to one shared legend
        ), row=row, col=col)

    fig.update_layout(
        title=f"Trading Strategy Grid Search for {ticker_symbol}",
        coloraxis=dict(
            colorscale=[[0, "red"], [0.5, "white"], [1, "green"]],
            cmin=vmin,
            cmax=vmax,
            cmid=0,
            colorbar=dict(title="Profit Above Baseline (%)")
        ),
        height=1800,
        width=1200
    )
    
    # Clean up tick density since we have 20 subplots
    fig.update_xaxes(nticks=10)
    fig.update_yaxes(nticks=10)
    
    fig.show()


# Stacked 3D Profit Heatmap
Visualizes the profit from the grid search across different wait periods as stacked 2D surfaces in a 3D plot.

In [ ]:
import numpy as np
import plotly.graph_objects as go

x_labels = [f"{x:.3f}" for x in sell_multipliers]
y_labels = [f"{y:.3f}" for y in buy_multipliers]

for ticker_symbol in backtested_profits_with_wait_periods.keys():
    fig = go.Figure()
    
    # Plot a surface layer for each wait period
    for w_idx, wait in enumerate(wait_periods):
        baseline_profit = baseline_profits[ticker_symbol]
        profits_matrix = (backtested_profits_with_wait_periods[ticker_symbol][wait] - baseline_profit) * 100 / baseline_profit
        
        # The Z-height for this surface is just the constant wait period, shape must match profits_matrix
        Z_surface = np.full_like(profits_matrix, wait)
        
        fig.add_trace(go.Surface(
            z=Z_surface,
            x=x_labels,
            y=y_labels,
            surfacecolor=profits_matrix, # We literally color the surface based on profit
            colorscale=[[0, "red"], [0.5, "white"], [1, "green"]],
            cmin=-10, # Hardcode roughly realistic bounds or calculate them dynamically
            cmax=10,
            opacity=0.8,
            name=f"Wait {wait}",
            showscale=(w_idx == 0) # Only show one colorbar for the whole plot
        ))

    fig.update_layout(
        title=f"Stacked 3D Profit Heatmap for {ticker_symbol}",
        scene=dict(
            xaxis_title='Sell Multipliers',
            yaxis_title='Buy Multipliers',
            zaxis_title='Wait Period (Hours)',
            # Z-axis should only show integers up to 20
            zaxis=dict(dtick=2)
        ),
        width=1000,
        height=800
    )

    fig.show()


# 3D Scatterplot Heatmap
Visualizes the profit from the grid search using a 3D scatterplot for true 3-axis representation.

In [ ]:
import numpy as np
import plotly.graph_objects as go

# TOGGLE: Set to True to only show profitable points, preventing the plot from looking like a solid cube
FILTER_PROFIT = True

Sell_mesh, Buy_mesh = np.meshgrid(sell_multipliers, buy_multipliers)

for ticker_symbol in backtested_profits_with_wait_periods.keys():
    fig = go.Figure()

    X, Y, Z, C = [], [], [], []
    
    for w_idx, wait in enumerate(wait_periods):
        baseline_profit = baseline_profits[ticker_symbol]
        profits_matrix = (backtested_profits_with_wait_periods[ticker_symbol][wait] - baseline_profit) * 100 / baseline_profit
        
        X.append(Sell_mesh.flatten())
        Y.append(Buy_mesh.flatten())
        Z.append(np.full(profits_matrix.size, wait))
        C.append(profits_matrix.flatten())

    X = np.concatenate(X)
    Y = np.concatenate(Y)
    Z = np.concatenate(Z)
    C = np.concatenate(C)
    
    if FILTER_PROFIT:
        mask = C > 0  # Only show strict profits over baseline
        X_plot, Y_plot, Z_plot, C_plot = X[mask], Y[mask], Z[mask], C[mask]
    else:
        X_plot, Y_plot, Z_plot, C_plot = X, Y, Z, C

    # Plot the 3D scatter 
    fig.add_trace(go.Scatter3d(
        x=X_plot,
        y=Y_plot,
        z=Z_plot,
        mode='markers',
        marker=dict(
            size=3,
            color=C_plot,                # set color to an array/list of desired values
            colorscale=[[0, "red"], [0.5, "white"], [1, "green"]],   # choose a colorscale
            cmid=0,
            opacity=0.6 if FILTER_PROFIT else 0.05,
            colorbar=dict(title="Profit Above Baseline (%)")
        ),
        # Nifty trick: Add hovering text to show the exact coordinates!
        text=[f"Profit: {c:.2f}%<br>Buy: {y:.3f}<br>Sell: {x:.3f}<br>Wait: {z}" \
              for x, y, z, c in zip(X_plot, Y_plot, Z_plot, C_plot)],
        hoverinfo='text'
    ))
    
    fig.update_layout(
        title=f"3D Scatter Heatmap for {ticker_symbol} \n({'Showing Only Profitable Configs' if FILTER_PROFIT else 'Showing All Configs'})",
        scene=dict(
            xaxis_title='Sell Multipliers',
            yaxis_title='Buy Multipliers',
            zaxis_title='Wait Period (Hours)'
        ),
        width=1000,
        height=800
    )

    fig.show()
